# Urban-v1 — entrenamiento controlado en Google Colab

Notebook para los cuatro experimentos independientes A2D2/UrbanSyn de `urban-exp-v1`: S-A2D2, S-UrbanSyn, I-UrbanSyn e I-A2D2. Usa el factory y runner del repositorio, guarda checkpoints y métricas en Google Drive y **nunca accede al split test**.

**Estado inicial seguro:** `RUN_TRAINING = False`. El protocolo exige cerrar auditorías, validar el materializado de datos y autorizar por separado el piloto y el entrenamiento completo. Esta notebook no descarga datasets ni sortea licencias: espera datos preparados en almacenamiento remoto autorizado. No descargues nada al teléfono.

Colab Free no garantiza GPU ni sesiones persistentes. Una sesión sin GPU se detiene antes de entrenar.

## 1. Configuración
El directorio `urban_data_v1` en Drive debe contener manifiestos locales, índices por imagen y assets materializados directamente desde fuentes autorizadas. La estructura exacta se valida más adelante.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib, time

REPO = 'Leohf04/Procesamiento-de-Im-genes-'
BRANCH = 'urban-models-and-benchmark'
REPO_URL = f'https://github.com/{REPO}.git'
WORKDIR = Path('/content/Procesamiento-de-Im-genes-')
DATA_ROOT = Path('/content/drive/MyDrive/ProcesamientoImagenes/urban_data_v1')
RESULTS = Path('/content/drive/MyDrive/ProcesamientoImagenes/experimentos/urban-exp-v1')
CONFIG_PATH = WORKDIR / 'configs/urban_experiment_v1.json'

print('Código:', WORKDIR)
print('Datos remotos persistentes:', DATA_ROOT)
print('Resultados:', RESULTS)


## 2. Montar Google Drive
Drive se usa para conservar datos ya preparados y resultados. No uses el almacenamiento del teléfono para los datasets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for p in [DATA_ROOT, RESULTS]: p.mkdir(parents=True, exist_ok=True)
print('Drive montado.')

## 3. Clonar la rama privada
Crea un secreto `GITHUB_TOKEN` en Colab Secrets con un Fine-grained PAT limitado a este repositorio y permiso `Contents: Read-only`. No pegues el token en el notebook ni en el chat.

In [ ]:
from getpass import getpass
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass('GitHub token (entrada oculta): ')
if not token: raise RuntimeError('Se necesita acceso de lectura al repositorio privado.')
askpass = Path('/tmp/git_askpass_urban.sh')
askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo x-access-token ;;\n*Password*) echo "$COLAB_GITHUB_TOKEN" ;;\n*) exit 1 ;;\nesac\n')
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','COLAB_GITHUB_TOKEN':token})
if WORKDIR.exists():
    subprocess.run(['git','-C',str(WORKDIR),'fetch','origin',BRANCH],check=True,env=env)
    subprocess.run(['git','-C',str(WORKDIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(WORKDIR),'pull','--ff-only','origin',BRANCH],check=True,env=env)
else:
    subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(WORKDIR)],check=True,env=env)
del token, env
os.chdir(WORKDIR)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

## 4. Dependencias adicionales y diagnóstico GPU
No reinstales PyTorch/Torchvision de Colab. Si no hay CUDA o Torchvision no importa, detén la ejecución.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','opencv-python-headless>=4.8,<5','pycocotools>=2.0.7','Pillow>=9','tqdm>=4.66','PyYAML>=6','matplotlib>=3.7','pytest>=7'],check=True)
import torch, torchvision
print('PyTorch:',torch.__version__,'Torchvision:',torchvision.__version__)
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Colab no asignó GPU. Cambia Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU y vuelve a ejecutar.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
subprocess.run(['nvidia-smi'],check=False)

## 5. Pruebas del código
Estas pruebas no entrenan modelos ni descargan datasets. Si fallan, no continúes.

In [ ]:
subprocess.run([sys.executable,'scripts/verify_setup.py'],check=True)
result = subprocess.run([sys.executable,'-m','pytest','-q'],text=True)
if result.returncode != 0:
    raise RuntimeError('La suite de pruebas falló. Detén el proceso y conserva la salida.')

## 6. Configuración experimental fijada
La celda lee `configs/urban_experiment_v1.json`. No modifica resolución, batch, optimizadores, acumulación, épocas ni semillas. A2D2 y UrbanSyn se entrenan por separado. SHIFT, Cityscapes y BDD100K no se incorporan a esta matriz.

In [ ]:
base_cfg = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assert base_cfg['experiment_protocol'] == 'urban-exp-v1'
assert base_cfg['ontology_version'] == 'urban-v1'
assert len(base_cfg['experiments']) == 4
print(json.dumps(base_cfg['experiments'],indent=2,ensure_ascii=False))
print('Política guardada en config:',base_cfg.get('run_policy'))

## 7. Validación de datos preparados (solo TRAIN y VAL)

Estructura requerida dentro de `urban_data_v1`:

```text
urban_data_v1/
  a2d2/class_list.json
  local_manifests/{a2d2,urbansyn}_{train,val}.jsonl
  annotation_index/{a2d2,urbansyn}_{train,val}.annotations.jsonl
  ...assets locales referenciados por image/semantic/instance/panoptic...
```

Los manifiestos locales deben tener rutas locales absolutas o relativas resolubles. A2D2 requiere `image` + `semantic` para semántica y `image` + `instance` para instancias; UrbanSyn requiere `image` + `semantic` o `panoptic`. Esta celda no abre ni consulta TEST.

In [ ]:
import json
MANIFEST_DIR = DATA_ROOT/'local_manifests'
INDEX_DIR = DATA_ROOT/'annotation_index'

def read_jsonl(path):
    return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]

def resolve_asset(value):
    p=Path(value)
    return p if p.is_absolute() else DATA_ROOT/p

checks=[]
for ds in ('a2d2','urbansyn'):
    train=read_jsonl(MANIFEST_DIR/f'{ds}_train.jsonl')
    val=read_jsonl(MANIFEST_DIR/f'{ds}_val.jsonl')
    for split,rows in [('train',train),('val',val)]:
        idx=read_jsonl(INDEX_DIR/f'{ds}_{split}.annotations.jsonl')
        ids=[str(r.get('sample_id','')) for r in rows]
        ann_ids=[str(r.get('sample_id','')) for r in idx]
        if not ids or any(not x for x in ids) or len(ids)!=len(set(ids)):
            raise RuntimeError(f'{ds}/{split}: sample_id vacío, duplicado o manifiesto vacío')
        if ids != ann_ids:
            raise RuntimeError(f'{ds}/{split}: IDs u orden de annotation_index no coinciden')
        if any(r.get('annotation_status','ok')!='ok' for r in idx):
            raise RuntimeError(f'{ds}/{split}: annotation_status distinto de ok')
        for row in rows:
            required=['image','semantic'] if ds=='urbansyn' else ['image','semantic']
            if ds=='a2d2': required=['image','semantic']
            for key in required:
                if key not in row or not resolve_asset(row[key]).is_file():
                    raise FileNotFoundError(f'{ds}/{split}: falta asset {key}: {row.get(key)}')
        checks.append({'dataset':ds,'split':split,'manifest_rows':len(rows),'index_rows':len(idx)})
    if set(str(x['sample_id']) for x in train) & set(str(x['sample_id']) for x in val):
        raise RuntimeError(f'{ds}: fuga train/val')
    checks.append({'dataset':ds,'train_val_overlap':0})

# Verificar también los assets específicos de cada tarea.
for ds,split,key in [('a2d2','train','instance'),('a2d2','val','instance'),('urbansyn','train','panoptic'),('urbansyn','val','panoptic')]:
    rows=read_jsonl(MANIFEST_DIR/f'{ds}_{split}.jsonl')
    missing=[r.get('sample_id') for r in rows if key not in r or not resolve_asset(r[key]).is_file()]
    if missing: raise FileNotFoundError(f'{ds}/{split}: {len(missing)} assets {key} ausentes; ejemplo {missing[:3]}')
if not (DATA_ROOT/'a2d2'/'class_list.json').is_file():
    raise FileNotFoundError('Falta a2d2/class_list.json')
print(json.dumps(checks,indent=2))
print('Datos TRAIN/VAL estructuralmente presentes. Esto no sustituye auditoría de licencia/calidad.')

## 8. Autorización y ejecución (bloqueada por defecto)

El protocolo vigente exige: (1) auditorías completas aprobadas; (2) materialización y validación de todos los assets; (3) piloto de 10 batches por configuración aprobado; y (4) autorización expresa del usuario para entrenamiento completo. No uses `TEST`.

**No cambies `RUN_TRAINING` a True todavía** si no has recibido confirmación de que las puertas previas están cerradas. La notebook comienza en modo seguro.

In [ ]:
RUN_TRAINING = False  # cambiar solo después de las autorizaciones requeridas
if RUN_TRAINING:
    # Confirmación adicional: los cuatro conjuntos de puertas deben estar marcados como aprobados.
    GATES = {
        'data_audit_approved': False,
        'license_and_storage_approved': False,
        '10_batch_pilot_approved_for_all_configs': False,
        'full_training_authorized': False,
    }
    if not all(GATES.values()):
        raise RuntimeError('BLOQUEADO: falta aprobar una o más puertas del protocolo. No se ha iniciado entrenamiento.')
else:
    print('Entrenamiento desactivado. Ningún modelo se entrenará en esta sesión.')

## 9. Lanzar los cuatro experimentos (solo tras desbloqueo)

Cada experimento se ejecuta de forma independiente, con su propia inicialización, optimizador, scheduler y carpeta. Se reanuda desde `last.pt` si existe. Los resultados se conservan en Drive. La ejecución no evalúa TEST.

In [ ]:
if RUN_TRAINING:
    for exp in base_cfg['experiments']:
        cfg={**base_cfg,**exp}
        cfg.pop('experiments',None)
        cfg['run_training']=True
        cfg['data_dir']=str(DATA_ROOT)
        cfg['annotation_index_dir']=str(INDEX_DIR)
        cfg['allow_test_access']=False
        exp_id=exp['id']
        cfg_path=RESULTS/exp_id/'config.json'
        out_dir=RESULTS/exp_id
        out_dir.mkdir(parents=True,exist_ok=True)
        cfg_path.write_text(json.dumps(cfg,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        command=[sys.executable,'scripts/run_urban_experiment.py','--config',str(cfg_path),
                 '--factory','scripts.urban_experiment_factory:create_experiment',
                 '--output-dir',str(out_dir),'--execute']
        last=out_dir/'last.pt'
        if last.is_file(): command += ['--resume',str(last)]
        print('\n===',exp_id,'===')
        subprocess.run(command,check=True)
else:
    print('No se ejecutó ningún entrenamiento. Mantén esta celda sin cambios hasta recibir autorización.')

## 10. Verificar y respaldar resultados

Al finalizar una sesión, comprueba que cada experimento tenga `last.pt`, `best.pt` y `metrics.jsonl`. Si Colab se desconecta, vuelve a montar Drive y ejecuta de nuevo la sección 9 con la misma configuración; el runner verificará el hash de configuración antes de reanudar.

In [ ]:
for exp in base_cfg['experiments']:
    p=RESULTS/exp['id']
    print(exp['id'],{
        'last.pt':(p/'last.pt').is_file(),
        'best.pt':(p/'best.pt').is_file(),
        'metrics.jsonl':(p/'metrics.jsonl').is_file(),
    })
print('No se evaluó TEST. No subas datos originales, máscaras ni checkpoints a un repositorio público.')